In [107]:
from pathlib import Path

from torchvision.transforms import v2
import torch

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib_inline.backend_inline as _
_.set_matplotlib_formats('svg')

from src.utils import COCO_CLASSES
from src.models import load_model
from src.data import COCODataset, get_dataloader
from src.engine import train_model
from src.utils import destandardize_image
from src.detection import detect_objects

In [108]:
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')

In [116]:
project_dir = Path('/Users/harshanurs/my_projects/GradDetect')

train_img_dir = project_dir / 'data/coco2017/train2017'
train_ann_file = project_dir / 'data/coco2017/annotations/instances_train2017.json'

val_img_dir = project_dir / 'data/coco2017/val2017'
val_ann_file = project_dir / 'data/coco2017/annotations/instances_val2017.json'

In [117]:
vgg_train_transform= v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.RandomHorizontalFlip(p=.5),
    v2.RandomAffine(degrees=10, translate=(0.1, 0.1)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

vgg_val_transform = v2.Compose([
    v2.ToImage(),
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [118]:
train_dataset = COCODataset(train_img_dir, train_ann_file, transform=vgg_train_transform)
val_dataset = COCODataset(val_img_dir, val_ann_file, transform=vgg_val_transform)

train_ldr = get_dataloader(train_dataset, batch_size=128, shuffle=True, drop_last=True)
val_ldr = get_dataloader(val_dataset, batch_size=64, shuffle=False, drop_last=True) # Keep it 64

In [ ]:
vgg = load_model('vgg', 1)
resnet = load_model('resnet', 1)

vgg, resnet

In [9]:
model, epoch_train_loss, epoch_train_metrics, epoch_val_loss, epoch_val_metrics = train_model(vgg, train_ldr, val_ldr, epochs=20, device=device)

 Epoch 20/20

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))

axs[0].plot(epoch_train_loss)
axs[0].plot(epoch_val_loss, color='red')
axs[0].set_title('Training Loss')
axs[0].set_xlabel('Epoch')
axs[0].set_ylabel('Loss')

axs[1].plot(epoch_train_metrics[:,0], label='Recall')
axs[1].plot(epoch_val_metrics[:,0], label='Val Recall')
axs[1].plot(epoch_train_metrics[:,1], label='Precision')
axs[1].plot(epoch_val_metrics[:,1], label='Val Precision')
axs[1].plot(epoch_train_metrics[:,2], label='F1')
axs[1].plot(epoch_val_metrics[:,2], label='Val F1')
axs[1].set_title('Training Metrics')
axs[1].set_xlabel('Epoch')
axs[1].set_ylabel('Metric')
axs[1].legend()

In [59]:
X, y = next(iter(val_ldr))

In [99]:
A = X[48]
B = y[48]

In [ ]:
plt.imshow(destandardize_image(A).permute(1, 2, 0).cpu())
plt.show()

In [ ]:
model = load_model('vgg')
model.load_state_dict(torch.load(project_dir / 'models' / 'VGG.pt'))
model.to(device)

In [ ]:
img = A.unsqueeze(0)

img, objs, predicted_class_ids, predicted_probs = detect_objects(img, model, threshold=0.6, device=device)

fig, ax = plt.subplots()

ax.imshow(img)

color = ['red', 'green', 'blue', 'yellow', 'purple', 'orange', 'black', 'brown', 'pink', 'gray', 'cyan', 'magenta', 'lime', 'teal', 'olive', 'navy', 'beige', 'maroon', 'turquoise', 'lavender', 'salmon', 'tan', 'snow', 'violet', 'gold', 'aqua', 'coral', 'plum', 'indigo', 'khaki', 'azure', 'limegreen', 'salmon', 'tan', 'snow', 'violet', 'gold', 'aqua', 'coral', 'plum', 'indigo', 'khaki', 'azure', 'limegreen']

for i, obj in enumerate(objs):

    x_min, y_min, x_max, y_max = obj
    width = x_max - x_min
    height = y_max - y_min

    rect = patches.Rectangle(
        (x_min, y_min),
        width,
        height,
        linewidth=1.5,
        edgecolor=color[i],
        facecolor="none",  # Transparent interior
        alpha=0.5
    )

    ax.add_patch(rect)
    ax.text(x_min, y_min-5, f'{COCO_CLASSES[predicted_class_ids[i].item()].capitalize()}:{predicted_probs[i].item():.2f}', color=color[i], fontsize=10, backgroundcolor='none')
